# Lesson 23 — Iterators: Practice (SUPPORTING)

Work in `lessons/lesson-23-iterators/` (point a terminal at that folder).

Before you start, read `lesson.md` — every construct used in these exercises is
taught there. Log lines are plain strings. Write each deliverable in its
stated file (R16 — no `solution.py`, no double-write), then run each
**Driver:** cell. Every `Expected` block must match byte-exact.

You create: `stream.py` (E1–M2) · `incidents.py` + `triage.py` (Challenge,
which imports `stream.py` — write once, reuse).
Provided: `buggy_triage.py` (buggy — H1b-style hunt, you fix it in place).
Installs: none — stdlib only (`collections.abc` taught in lesson.md).


## E1 — Pull the first N without storing the stream

## Intent
**Pull the first N items from any stream as a list — never holding more.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `stream.py`, define `take(stream: Iterable[str], n: int) -> list[str]`.
Pull up to `n` items via `iter()` + `next()` inside `try/except
StopIteration` (the sentinel-free form from the lesson — graded code never
conflates a real item with exhaustion). A short stream yields whatever
exists; `n == 0` yields `[]`.

**File tree**

- `stream.py` (`take`)

**Interface contract**

- `take(stream, n)` returns the pulled items in order, unchanged.

**Constraints**

- **`n < 0` raises `ValueError("take needs n >= 0")`.**
- **Works on any iterable** — lists, iterators, file handles.
- **Pull at most `n` times** (a capped stream must survive — no hoarding).

**Coverage**

- `iter`/`next`/`StopIteration` → lesson.md Core Concept 1.

**Expected** (Driver cell):
```text
E1 OK
```


In [ ]:
import io
import stream as s

assert s.take(iter([1, 2, 3]), 2) == [1, 2]
assert s.take([1], 5) == [1]
assert s.take(iter([1]), 0) == []
assert s.take(iter([]), 3) == []
try:
    s.take([1], -1)
except ValueError as e:
    assert str(e) == "take needs n >= 0"
else:
    raise AssertionError("no raise")
assert s.take(io.StringIO("a\nb\n"), 5) == ["a\n", "b\n"]
print("E1 OK")

## E2 — Tell re-readable sources from one-shot cursors

## Intent
**Decide on sight whether a stream survives a second pass.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `stream.py`, define `kind(stream) -> str`. Return `"iterator"` when
`iter(stream) is stream`, else `"iterable"`. That identity check is the
whole implementation — and the whole point of lesson.md Core Concept 2.

**File tree**

- `stream.py` (`kind`)

**Interface contract**

- `kind(stream)` returns exactly `"iterator"` or `"iterable"`.

**Constraints**

- **The identity check is the decision procedure — no type lists.**
- **No consumption** (deciding must not pull a single item).

**Coverage**

- Iterable vs iterator → lesson.md Core Concept 2.

**Expected** (Driver cell):
```text
E2 OK
```


In [ ]:
import io
import stream as s

assert s.kind([1, 2]) == "iterable"
assert s.kind({"a": 1}) == "iterable"
assert s.kind(iter([1])) == "iterator"
assert s.kind(map(str, [1])) == "iterator"
assert s.kind(io.StringIO("x")) == "iterator"
h = io.StringIO("x\ny\n")
before = h.tell()
s.kind(h)
assert h.tell() == before
print("E2 OK")

## E3 — Validate a stream at the boundary

## Intent
**Reject rotten log lines where they enter — and survive a dry stream.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `stream.py`, define `validate_line(line: str) -> str` and
`first_valid(stream)`. `validate_line` returns the stripped line; a
non-string raises `ValueError("invalid line: not a string")`, a
blank-after-strip raises `ValueError("invalid line: empty")` (check in
that order). `first_valid` skips invalid lines silently and returns the
first valid stripped line — or `None` when the stream holds none,
including an exhausted one-shot iterator (no crash, no sentinel leak).

**File tree**

- `stream.py` (`validate_line`, `first_valid`)

**Interface contract**

- `validate_line(line)` returns stripped text or raises (exact strings).
- `first_valid(stream)` returns stripped text or `None`.

**Constraints**

- **Exact messages verbatim, checked in the listed order.**
- **`first_valid` pulls once through** (works on consumed-one-shot input).

**Coverage**

- Boundary validation + exhaustion edge → lesson.md Core Concept 3 +
  Engineering Notes (validate per item).

**Expected** (Driver cell):
```text
E3 OK
```


In [ ]:
import importlib
import stream as s; importlib.reload(s)

assert s.validate_line("  ok  ") == "ok"
for bad, msg in [(None, "invalid line: not a string"),
                 (123, "invalid line: not a string"),
                 ("", "invalid line: empty"),
                 ("   \n", "invalid line: empty")]:
    try:
        s.validate_line(bad)
    except ValueError as e:
        assert str(e) == msg, (bad, str(e))
    else:
        raise AssertionError(("no raise", bad))
assert s.first_valid(["  ", "", "ok "]) == "ok"
assert s.first_valid([]) is None
assert s.first_valid(iter([])) is None
assert s.first_valid(iter(["", "  x"])) == "x"
print("E3 OK")

## M1 — Batch any stream into fixed-size windows

## Intent
**Turn any stream into an iterator of fixed-size windows — ragged tail kept.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `stream.py`, define the `WindowReader` class: `__init__(self, source,
size)` stores `iter(source)` as a private cursor after validating `size`;
`__iter__` returns `self` (one-shot reader, stated in a comment);
`__next__` pulls up to `size` items, emitting the partial tail as a real
window, and raises `StopIteration` when the source is dry before a
window's first pull. Annotate with `collections.abc` names as taught
(`Iterator[list[str]]` behavior).

**File tree**

- `stream.py` (`WindowReader`)

**Interface contract**

- `list(WindowReader(source, size))` is the window list; empty source →
  zero windows; oversized `size` → one window.

**Constraints**

- **Non-int or `< 1` size raises `ValueError("window size must be >= 1")`.**
- **The ragged tail is a real window, never dropped.**
- **One-shot: a second full pass yields `[]`.**
- **Cursor lives on the instance, never the class.**

**Coverage**

- Custom iterator class → lesson.md Core Concept 5 (+ §2 annotations).

**Expected** (Driver cell):
```text
M1 OK
```


In [ ]:
import io
import importlib
import stream as s; importlib.reload(s)

assert list(s.WindowReader([1, 2, 3, 4, 5], 2)) == [[1, 2], [3, 4], [5]]
assert list(s.WindowReader([1, 2], 1)) == [[1], [2]]
assert list(s.WindowReader([], 3)) == []
assert list(s.WindowReader([1, 2, 3], 9)) == [[1, 2, 3]]
for bad_size in (0, -2, "2"):
    try:
        s.WindowReader([1], bad_size)
    except ValueError as e:
        assert str(e) == "window size must be >= 1"
    else:
        raise AssertionError(("no raise", bad_size))
h = io.StringIO("a\nb\nc\n")
assert list(s.WindowReader(h, 2)) == [["a\n", "b\n"], ["c\n"]]
r = s.WindowReader([1, 2, 3], 2)
assert list(r) == [[1, 2], [3]]
assert list(r) == []
print("M1 OK")

## M2 — Compose validate-then-batch yourself

## Intent
**Build the valid-lines pipeline from your own pieces — order observable.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `stream.py`, define `clean_windows(source, size)` using YOUR
`validate_line` and `WindowReader` (this composition is the exercise —
the driver checks behavior only). Drop invalid lines FIRST so blanks
never occupy window slots, then batch what survived. Small driver data
may materialize the clean list; say so in one comment (the Challenge
streams for real — laziness posture is graded there, honesty here).

**File tree**

- `stream.py` (`clean_windows` — no new file)

**Constraints**

- **Validate-first: blanks never occupy window slots.**
- **Reuse your own tools (no reimplemented validation, no second
  batcher).**
- **`size` violations surface the M1 `ValueError` unchanged.**

**Coverage**

- Learner-authored composition → M2 law + Core Concepts 3/5.

**Expected** (Driver cell):
```text
M2 OK
```


In [ ]:
import importlib
import stream as s; importlib.reload(s)

got = s.clean_windows(["a", "", "b", "  ", "c", "d"], 2)
assert got == [["a", "b"], ["c", "d"]], got
assert s.clean_windows(["", "  "], 2) == []
assert s.clean_windows([], 3) == []
try:
    s.clean_windows(["a"], 0)
except ValueError as e:
    assert str(e) == "window size must be >= 1"
else:
    raise AssertionError("no raise")
print("M2 OK")

## H1 — Hunt: four stream bugs in buggy_triage

## Intent
**Diagnose four real iterator failures in a provided module — fix each where it lives.**

**Type:** DEBUG — Hard — fixes 1 file

**Task**
`buggy_triage.py` is provided, buggy: four planted failures, one per
Pitfall (truthiness guard, swallowed termination, consumed reuse, shared
cursor). Nothing else in the file is wrong. Run the DEBUGGING protocol:
reproduce via the driver, read the traceback, locate, hypothesize, fix,
explain. Fix in place; do not restructure the module.

**File tree**

- `buggy_triage.py` (`first_error`, `StuckReader`, `count_and_errors`,
  `SharedReader` — YOU fix all four)

**Interface contract** (fixed states, exact)

- `first_error` scans and returns the first ERROR line stripped, else
  `None` — on lists and one-shot iterators alike.
- `StuckReader` terminates: a 3-item source yields exactly 3 items.
- `count_and_errors` returns `(total, error_lines)` with a non-empty
  body after counting.
- Two `SharedReader` instances advance independently.

**Constraints**

- **Four fixes only — no refactoring, no new symbols.**
- **Each fix stays in its own layer.**
- **Fill one 7-row chain per bug in the next cell, as you fix.**

**Coverage**

- The four stream failures → lesson.md Pitfalls 1–4.

**Expected** (Driver cell):
```text
H1 OK
```


### H1 chain reports (one per bug — fill as you fix)

#### Bug 1 — first_error
| Step | Content |
|---|---|
| Symptom | AssertionError: assert bt.first_error(["ok", "ERROR: disk full"]) == "ERROR: disk full" |
| Trigger | buggy_triage.first_error |
| Hypothesis | no loop on stream |
| Evidence | fail to catch first error on second item provided list |
| Cause | direct return on first item of iter |
| Fix | add loop through stream |
| Proof | assertion passed |

#### Bug 2 — StuckReader
| Step | Content |
|---|---|
| Symptom | AssertionError: assert list(islice(bt.StuckReader(["a", "b", "c"]), 10)) == ["a", "b", "c"] |
| Trigger | buggy_triage.StuckReader |
| Hypothesis | the result is containing None for the rest item in length N |
| Evidence | ['a', 'b', 'c', None, None, None, None, None, None, None] |
| Cause | StopIteration exception return None |
| Fix | direct return next(self._cursor) so that the StopIteration auto raised |
| Proof | ['a', 'b', 'c'] |

#### Bug 3 — count_and_errors
| Step | Content |
|---|---|
| Symptom | AssertionError: assert (n, errs) == (3, ["ERROR: x"]) |
| Trigger | buggy_triage.count_and_errors |
| Hypothesis | consumed cursor |
| Evidence |     total = sum(1 for _ in it) \n errors = [line for line in it if "ERROR" in line] |
| Cause | reuse the iterator object |
| Fix | redeclare or reuse the iterable |
| Proof | 3 ['ERROR: x'] |

#### Bug 4 — SharedReader
| Step | Content |
|---|---|
| Symptom | AssertionError: assert next(r1) == [1] |
| Trigger | buggy_triage.SharedReader |
| Hypothesis | using the Sharedreader._cursor instead of self |
| Evidence | SharedReader._cursor = iter(source) \n SharedReader._size = size |
| Cause | set the last call source as the iter for whoever call it |
| Fix | use self instead |
| Proof | H1 OK |


In [ ]:
import importlib
import buggy_triage as bt
importlib.reload(bt)
from itertools import islice

assert bt.first_error(["ok", "ERROR: disk full"]) == "ERROR: disk full"
assert bt.first_error(iter(["ok", "ERROR: x"])) == "ERROR: x"
assert bt.first_error(["all good"]) is None
assert list(islice(bt.StuckReader(["a", "b", "c"]), 10)) == ["a", "b", "c"]
n, errs = bt.count_and_errors(["a", "ERROR: x", "b"])
assert (n, errs) == (3, ["ERROR: x"])
r1 = bt.SharedReader([1, 2, 3], 1)
r2 = bt.SharedReader([9, 8], 1)
assert next(r1) == [1]
assert next(r2) == [9]
assert next(r1) == [2]
print("H1 OK")

## Challenge — Night-shift triage run for the log

## Intent
**Ship a triage run over the server log: every ERROR line reported, every rotten line skipped loudly — in constant-memory posture.**

**Type:** REAL-WORLD — Micro Build — writes 2 files

**Task**
The on-call engineer runs one command before dawn:
`python triage.py <server.log> <incidents.txt>`. It streams the log
through windows, validates each line, collects ERROR lines, skips rotten
ones with a loud `skip` line, and writes an incidents report that is also
printed — same bytes both places.

Create `incidents.py` and `triage.py` in this folder. Reuse `stream.py`
(`WindowReader`, `validate_line` — write once, import, never double-write).

**Flow** (the whole wiring — every arrow is required)

```text
main reads <server.log> as a streaming handle (never read_text for the log)
  → WindowReader(enumerate(handle), 3): numbered lines in windows of 3
  → per (i, raw) in file order:
      text = validate_line(raw)            # rotten → skip line, no error check
      "ERROR" in text → error line         # valid non-errors produce no line
  → text = build_incidents(lines)
  → write text to <incidents.txt> AND print the same text to stdout
```

The log is always a valid text file — no corrupt-file handling required.
"Missing log" means the path does not exist or is not a file.

**File tree**

- `incidents.py` (`build_incidents`)
- `triage.py` (`main`)

**Interface contract**

- `incidents.build_incidents(lines)` returns header + lines + trailing
  newline (header alone when empty). It performs no I/O.
- Header shape: `errors E, skipped S` (counts derived from line
  prefixes `error ` / `skip `).
- `triage.main(argv)` — house shape (L20–L22): full `sys.argv`
  (`main(sys.argv)` guard), paths at `argv[1]` / `argv[2]`, `-> None`,
  errors leave via `sys.exit(1)` after one stderr line.

**Constraints**

- **Consume the handle ONLY through `WindowReader`** (never iterate it
  directly — laziness posture is review-graded).
- **Per line: validate → error-check → error/skip line; valid
  non-errors emit nothing.**
- **Wrong argc prints `usage: triage.py <server.log> <incidents.txt>`
  to stderr and exits 1.**
- **Missing log prints `bad log: not found` to stderr and exits 1.**
- **Stdout and the incidents file are byte-identical.**

**Coverage**

- Streaming + batching + boundary validation composed → lesson.md Core
  Concepts 1–6 + house entry (L20–L22 retrieval).

**Expected** (Driver cell, real subprocess runs, fixture of 6):
```text
CH OK
```


In [ ]:
import subprocess, sys, tempfile, pathlib
HOME = pathlib.Path.cwd()
log_lines = ["INFO boot ok", "ERROR disk full", "", "INFO retrying",
             "ERROR net down", "INFO shift end"]
expected = ("errors 2, skipped 1\n" "error 1: ERROR disk full\n"
    "skip 2: invalid line: empty\n" "error 4: ERROR net down\n")
with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    lg, rp = td / "server.log", td / "incidents.txt"
    lg.write_text("\n".join(log_lines) + "\n", encoding="utf-8")
    run = subprocess.run(
        [sys.executable, "triage.py", str(lg), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert run.returncode == 0, run.stderr
    assert run.stdout == expected, repr(run.stdout)
    assert rp.read_text(encoding="utf-8") == expected
    usage = subprocess.run(
        [sys.executable, "triage.py", str(lg)],
        capture_output=True, text=True, cwd=HOME)
    assert usage.returncode == 1
    assert usage.stderr == "usage: triage.py <server.log> <incidents.txt>\n"
    missing = subprocess.run(
        [sys.executable, "triage.py", str(td / "nope.log"), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert missing.returncode == 1
    assert missing.stderr == "bad log: not found\n"
print("CH OK")